# LLM-Based Scientific Paper Summarization

## Project Overview

This practice project is inspired by the [What's Up, Docs? competition](https://www.drivendata.org/competitions/297/whats-up-docs/) hosted on DrivenData. The objective is to develop a language model capable of generating concise, accurate, and informative summaries of scientific papers published on **SocArXiv**.

The project focuses on exploring and comparing different approaches to **transformer-based text summarization**, with an emphasis on understanding model architectures, training techniques, and parameter-efficient fine-tuning.

## Project Objectives

Two main approaches will be implemented and evaluated:

1. **Transformer From Scratch:** Design, implement, and train a small Transformer model from scratch to gain a deeper understanding of attention mechanisms, model architecture, and the training process.

2. **Parameter-Efficient Fine-Tuning (PEFT):** Fine-tune a pretrained small language model using two techniques:
   - **LoRA (Low-Rank Adaptation):** Adapt a pretrained model by training a small number of additional parameters.
   - **QLoRA (Quantized Low-Rank Adaptation):** Combine quantization with LoRA to reduce GPU memory requirements during fine-tuning.

## Expected Outcomes

The goal is to compare these approaches in terms of **summarization quality, computational efficiency, GPU memory consumption, and training performance**, while gaining hands-on experience with modern NLP techniques.

## Import requiered Libraries

In [1]:
import sys
import os
from pathlib import Path
from google.colab import drive


drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""!cp /content/drive/MyDrive/projects/Document_Summarization/data.zip /content/
!unzip -q /content/data.zip -d /content/data"""

In [3]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/projects/Document_Summarization"
)

PROJECT_DATA_DIR = Path("/content/data")

DATA_DIR = PROJECT_DATA_DIR / "data"
SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print(os.listdir(os.path.join(PROJECT_ROOT, "")))

['src', 'data', 'data.zip']


In [ ]:

# data processing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
# machine learning libraries
import torch
from torch import nn as nn
from torch import optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

import math
import copy

# Hugging Face libraries

from transformers import AutoTokenizer, AutoModelForCausalLM


In [5]:
import random

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cpu
CUDA version: None
CUDA available: False


## Extract and View data

In [7]:

SUBMISSION_FORMAT_PATH = Path.joinpath(DATA_DIR, "submission_format.csv")
TRAIN_PATH = Path.joinpath(DATA_DIR, "train.csv")
TEST_FEATURES_PATH = Path.joinpath(DATA_DIR, "test_features.csv")


for path in [SUBMISSION_FORMAT_PATH, TRAIN_PATH, TEST_FEATURES_PATH]:
    print(f"{path.name}: {path.exists()}")


print("✅ All datasets loaded successfully!")

submission_format.csv: True
train.csv: True
test_features.csv: True
✅ All datasets loaded successfully!


In [8]:
submission_format_df = pd.read_csv(SUBMISSION_FORMAT_PATH)
train_df = pd.read_csv(TRAIN_PATH)
test_features_df = pd.read_csv(TEST_FEATURES_PATH)


In [9]:
train_df.head()

,paper_id,text,summary
0,0,## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCI...,"In this article, Victor Fan argues that analys..."
1,1,## 1. Introduction\n\n\nAn Electronic Health R...,Problem definition: Physicians spend more than...
2,2,## Introduction\n\n\nTranslation plays an i...,Literary translation is one of the most challe...
3,3,## 1 Problem Setup\n\n\nRecent political scien...,There is a long-running debate on evaluating f...
4,4,## INTRODUCTION\n\n\nThis article investigat...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."


In [10]:
print("Lenght of the submission",len(submission_format_df))
display(submission_format_df.head(3))

Lenght of the submission 345


,paper_id,summary
0,1000,my very accurate summary
1,1001,my very accurate summary
2,1002,my very accurate summary


In [11]:
test_features_df.head(3)

,paper_id,text
0,1000,## Introduction\n\n\nGender disparities persis...
1,1001,## Introduction\n\n\nOne of humanity’s greates...
2,1002,## Introduction\n\n\nHow do workers get attach...


In [12]:
print("Length if the text : " , len(train_df.iloc[0].text))

print(textwrap.fill(train_df.iloc[0].text[:500], width=100))



Length if the text :  33416
## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCIOUSNESS   Since 1997, the concept of
extraterritoriality has been configured in the political tension between Hong Kong and Beijing. From
the perspective of the Central Government, it is fundamental for the people of China to shijian
zhuquan instantiate its sovereignty over Hong Kong. But while most Hong / Kong residents insist on
interpreting this concept in terms of the Euro-American notion of selfdetermination  (zizhu / making
decisions  for  ones


In [13]:
print("Length of the summary : " ,len(train_df.iloc[0].summary))
print(textwrap.fill(train_df.iloc[0].summary[:500], width=100))


Length of the summary :  1302
In this article, Victor Fan argues that analysing contemporary Hong Kong cinema requires active
rewriting of established postcolonial theories by taking into account the specific mode of
colonisation of Hong Kong: extraterritoriality. This concept has been responsible for the
construction of the cultural plurality, linguistic ambiguity, and political liminality of Hong Kong
and its cinematographic experience, as well as the incongruence between the community’s political
consciousness after 1997


In [14]:
print("Length if the text : " , len(test_features_df.iloc[0].text))

print(textwrap.fill(test_features_df.iloc[0].text[:500], width=100))



Length if the text :  47034
## Introduction   Gender disparities persist in how men and women divide paid work, housework and
childcare responsibilities (Cotter et al., 2011; England, 2010; Kühhirt, 2012). Despite rising
levels of education and labor force participation among women, they continue to perform a larger
portion of housework and childcare compared to their male partners (Auspurg et al., 2017; Schulz,
2021). This gendered division of labor remains prevalent, even as traditional views on gender roles
have decline


## Preprocessing

## Build small LLM model

In [ ]:
class textDataset(Dataset):
    def __init__(self , df : pd.DataFrame , transform):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.df[idx]
        summary = self.df[idx]

        if self.transform is not None:
            text = self.transform(text)
            summary = self.transform(summary)

        return text , summary

def collate(batch):
    pass

In [ ]:


train_loader = DataLoader()

val_loader = DataLoader()


In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        pass

    def forward(self,x):
        pass

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        pass

    def forward(self,x):
        pass

In [ ]:
class Transformer(nn.Module):
    def __init__(self):
        pass

    def forward(self,x):
        pass

## LoRa & QLoRa